### Human In The Loop

The Human-in-the-Loop (HITL) middleware lets you add human oversight to agent tool calls. When a model proposes an action that might require review—for example, writing to a file or executing SQL—the middleware can pause execution and wait for a decision.

It does this by checking each tool call against a configurable policy. If intervention is needed, the middleware issues an interrupt that halts execution. The graph state is saved using LangGraph’s persistence layer, so execution can pause safely and resume later.

A human decision then determines what happens next: the action can be approved as-is (approve), modified before running (edit), rejected with feedback (reject), or responded to directly (respond) for “ask user” style tools.

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langgraph.types import Command

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [14]:
agent = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                #tool on which the agent has to ask for the human approval
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [25]:
config = {"configurable": {"thread_id": "test-approve"}}

#step 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'")
        ]
    },
    config = config
)

In [23]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='c19f3732-3cde-4d99-bd88-d33ae21d1aea'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "recipient": "john@test.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'call_12390866644103289484': 'Eo4BCosBAWkUfRPnMbuCBMBjlcETBOqsmoT4IXh/P9LucQXxtJ7og0zb1jJfHxSWIE1ooLAj+C2wYr1TKogpwUMq3u98rl49ybve9vC5nMsM84GlXEZ4n3pq+gw7MsuRd+gua2Dl9GZjfd2TbQFsO/F+KE/UKamfpqNkXoz2LQevCkCeniT3vLPVcwHy4NoPaw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1fe78-4a43-708d-99d1-1d566e961212-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'how are you?', 'recipient': 'john@test.com', 'subject': 'hello'}, 'id': 'call_1239086664410

In [22]:
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ), 
        config = config
    )
    print(f"✅ Result: {result['messages'][-1].content}")

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='c19f3732-3cde-4d99-bd88-d33ae21d1aea'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "recipient": "john@test.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'call_12390866644103289484': 'Eo4BCosBAWkUfRPnMbuCBMBjlcETBOqsmoT4IXh/P9LucQXxtJ7og0zb1jJfHxSWIE1ooLAj+C2wYr1TKogpwUMq3u98rl49ybve9vC5nMsM84GlXEZ4n3pq+gw7MsuRd+gua2Dl9GZjfd2TbQFsO/F+KE/UKamfpqNkXoz2LQevCkCeniT3vLPVcwHy4NoPaw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1fe78-4a43-708d-99d1-1d566e961212-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'how are you?', 'recipient': 'john@test.com', 'subject': 'hello'}, 'id': 'call_1239086664410

### Reject

In [ ]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Rejecting...")
    
    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ), 
        config = config
    )
    print(f" Result: {result['messages'][-1].content}")

⏸️ Paused! Rejecting...
 Result: [{'type': 'text', 'text': "The tool call was rejected, so I did not send the email. Let me know if you'd like to try again or need help with something else!", 'extras': {'signature': 'Eo4BCosBAWkUfRNxCfw1ke197TOEuehNw/OP0A8PZbwHCTBMPzZYQWigRADQKUWSeFj2q/h089/xSKE4Qlm9oCHdz9/s1Wb6MufaHw3S0hD1UClin/o+Soo60PYeuT6qD/pjvPv+OKmtsLBytABRyykyWmc+2kFceMMYnTjMKSShE8WovzTfOyzi9F84BQgTzw=='}}]


In [27]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='c19f3732-3cde-4d99-bd88-d33ae21d1aea'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "recipient": "john@test.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'call_12390866644103289484': 'Eo4BCosBAWkUfRPnMbuCBMBjlcETBOqsmoT4IXh/P9LucQXxtJ7og0zb1jJfHxSWIE1ooLAj+C2wYr1TKogpwUMq3u98rl49ybve9vC5nMsM84GlXEZ4n3pq+gw7MsuRd+gua2Dl9GZjfd2TbQFsO/F+KE/UKamfpqNkXoz2LQevCkCeniT3vLPVcwHy4NoPaw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1fe78-4a43-708d-99d1-1d566e961212-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'how are you?', 'recipient': 'john@test.com', 'subject': 'hello'}, 'id': 'call_1239086664410